# Insurance claim review — Gemini API

Set `GEMINI_API_KEY` in the cell below, then run all cells.

In [ ]:
!pip install -q google-generativeai pandas

In [ ]:
import os
import sys
from pathlib import Path

def find_project_root() -> Path:
    """Find folder containing data/claims.csv under cwd or /home/cdsw."""
    for start in [Path.cwd(), Path("/home/cdsw"), Path.home()]:
        if not start.exists():
            continue
        if (start / "data" / "claims.csv").exists():
            return start
        for claims_csv in start.rglob("claims.csv"):
            if claims_csv.parent.name == "data":
                return claims_csv.parent.parent
    raise FileNotFoundError(
        "Upload insurance-demo to CAI (need data/claims.csv). "
        "Run: !ls -la /home/cdsw"
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("claims.csv:", (PROJECT_ROOT / "data/claims.csv").exists())
print("policy.md:", (PROJECT_ROOT / "data/policy/coverage_policy.md").exists())

In [ ]:
os.environ["GEMINI_API_KEY"] = "paste-your-key-here"
os.environ["GEMINI_MODEL"] = "gemini-2.5-pro"

In [ ]:
import json
import google.generativeai as genai
from agent.gemini_adjudicator import load_claim_features, SYSTEM_PROMPT

genai.configure(api_key=os.environ["GEMINI_API_KEY"])

claim_id = "CLM-50021"
features = load_claim_features(claim_id)
policy = (PROJECT_ROOT / "data" / "policy" / "coverage_policy.md").read_text()

model = genai.GenerativeModel(
    os.environ.get("GEMINI_MODEL", "gemini-2.5-pro"),
    system_instruction=SYSTEM_PROMPT,
)
response = model.generate_content(
    f"Review this claim.\n\nFeatures:\n{json.dumps(features, indent=2)}\n\nPolicy:\n{policy}"
)
print(response.text)